# core

> Add/remove a GPU to/from your SolveIt instance in one function.

```mermaid
flowchart TD
    A[Provision Modal sandbox]
    B[Setup SSH connection]
    C[Swap IPython kernel]

    A --> B --> C
```

In [ ]:
#| default_exp core

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
enable_mermaid()

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

---

We first have to provision a sandbox.

In [ ]:
#| export
from solveit_modal.sandbox import *

In [ ]:
#| export
def provision_sandbox(
    app_name:str,   # App name to look up or create
    pips    :list,  # pip packages to have installed
    apts    :list,  # apt packages to have installed
    vol_name:str,   # Volume name to look up or create
    timeout :int,   # Auto-terminate sandbox after this many seconds
    gpu     :str,   # GPU type (e.g, 'T4', 'A100')
    secrets:dict,   # Sandbox secrets
) -> tuple:  # (sandbox, host, port)
    app = mk_app(app_name)
    img = mk_img(pips, apts)
    vol = mk_vol(vol_name)
    sb = mk_sandbox(app, img, vol, timeout, gpu, secrets)
    host,port = get_tunnel(sb)
    return sb,host,port

In [ ]:
#| export
default_pips = [
    'ipykernel',
    'fastai', 'transformers', 'diffusers', 'accelerate', 'datasets',
    'huggingface_hub', 'peft', 'safetensors', 'sentence-transformers',
    'xformers', 'bitsandbytes', 'ninja', 'einops',
    'wandb',
    'gradio', 'python-fasthtml', 'plotly', 'ipywidgets',
    'fsspec', 's3fs', 'gcsfs',
    'librosa', 'imageio', 'imageio-ffmpeg',
]

In [ ]:
#| export
import json

In [ ]:
#| export
def get_secrets() -> dict[str,str]:
    "Secrets from solveit_settings.json."
    with open('/app/data/solveit_settings.json') as f: return json.load(f)['secrets']

To make things as seamless as possible, I'll be adding the secrets in this SolveIt instance to the Modal sandbox.

In [ ]:
#| export
import modal

In [ ]:
sb, host, port = provision_sandbox(app_name='solveit-sandbox', pips=default_pips, apts=['openssh-server'], vol_name='solveit-sandbox', timeout=600, gpu='T4', secrets=get_secrets()); sb, host, port

INFO - ∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... | 2026-06-17 06:58:11,132


INFO - ✔ sandbox ready | 2026-06-17 06:58:11,651


INFO - ✔ gotten tunnel: r448.modal.host:40691 | 2026-06-17 06:58:25,188


(Sandbox(), 'r448.modal.host', 40691)

With the sandbox provisioned, it's now time to set up the SSH connection.

In [ ]:
#| export
from typing import Callable

In [ ]:
#| export
def setup_ssh(
    sb  :modal.sandbox.Sandbox,  # Modal Sandbox object
    host:str,                    # Tunnel hostname
    port:int                     # Tunnel port
) -> Callable:  # SSH connection
    "Inject public key, start SSH service, and verify connectivity."
    inject_pubkey(sb, get_pubkey())
    start_ssh(sb)
    ssh = mk_ssh(host, port)
    verify_ssh(ssh)
    return ssh

In [ ]:
ssh = setup_ssh(sb, host, port); ssh

INFO - ✔ public key injected | 2026-06-17 06:58:35,425


INFO - ✔ started ssh service | 2026-06-17 06:58:36,090


System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: Tesla T4


<function solveit_modal.sandbox.mk_ssh.<locals>.<lambda>(*cmd)>

In [ ]:
ssh('hostname')

'modal'

The last step is to swap the IPython kernel.

In [ ]:
#| export
from solveit_modal.kernel import *

In [ ]:
#| export
from ipyfernel.core import *

In [ ]:
#| export
_all_ = ['connect_existing_kernel', 'gip', 'ipf_exec', 'ipf_shutdown', 'ipf_startup', 'local', 'register_remote_kernel', 'remote', 'set_ssh_config', 'set_sticky', 'start_remote', 'stop_remote', 'unset_sticky']

I'd do `_all_ = ipyfernel.core.__all__`, but nbdev would through an error.

In [ ]:
#| export
import logging

In [ ]:
#| export
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s | %(asctime)s')
log = logging.getLogger(__name__)

In [ ]:
#| export
def link_remote_kernel(
    ssh   :object,     # Paramiko SSH connection
    host  :str,        # Tunnel hostname
    port  :int,        # Tunnel port
    sticky:bool=False  # Set `True` to execute code remotely by default
):
    "Launch remote IPython kernel on sandbox and connect ipyfernel to it."
    start_remote_kernel(ssh)
    p = get_remote_python(ssh)
    register_remote_kernel(remote_python=p)
    connect_existing_kernel(f'{host}:{port}')
    log.info('✔ connected to remote kernel')
    if sticky: 
        set_sticky()
        log.warning('! code now runs remotely by default; run `%local unset_sticky()` to run code locally')
    log.warning('! remote kernel environment has a different set of libraries installed')

In [ ]:
link_remote_kernel(ssh, host, port)

INFO - ∞ starting kernel | 2026-06-17 06:59:28,756


INFO - ✔ remote kernel ready: /root/.local/share/jupyter/runtime/kernel-ipyf.json | 2026-06-17 06:59:33,555


ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


INFO - ✔ connected to remote kernel | 2026-06-17 06:59:35,452


WARNING - ! remote kernel environment has a different set of libraries installed | 2026-06-17 06:59:35,453


Success: connected to remote kernel via r448.modal.host:40691


And we're done!

In [ ]:
%remote !hostname

modal


In [ ]:
stop_remote()

Code cells will now run locally.
Shutting down remote kernel


In [ ]:
sb.terminate()

I'll wrap everything up into a single function so we can attach a GPU in one line.

In [ ]:
#| export
def gpu_on(
    app_name:str='solveit-modal',      # App name to look up or create
    pips    :list=default_pips,        # pip packages to have installed
    apts    :list=['openssh-server'],  # apt packages to have installed
    vol_name:str='solveit-volume',     # Volume name to look up or create
    timeout :int=1800,                 # Auto terminate sandbox after this many seconds
    gpu     :str='T4',                 # GPU type (e.g., 'T4', 'A100')
    secrets :dict|None=None,           # Sandbox secrets
    sticky  :bool=False                # Set `True` to execute code remotely by default
) -> tuple:  # (sandbox, ssh)
    "Provision a GPU sandbox, wire up SSH, and hijack cells onto a remote kernel."
    sb,host,port = provision_sandbox(app_name,pips,apts,vol_name,timeout,gpu,secrets=get_secrets() if secrets is None else secrets)
    ssh = setup_ssh(sb,host,port)
    link_remote_kernel(ssh,host,port,sticky=sticky)
    return sb,ssh

In [ ]:
sb, _ = gpu_on()

INFO - ∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... | 2026-06-17 07:00:27,183


INFO - ✔ sandbox ready | 2026-06-17 07:00:27,637


INFO - ✔ gotten tunnel: r450.modal.host:44339 | 2026-06-17 07:00:43,028


INFO - ✔ public key injected | 2026-06-17 07:00:43,430


INFO - ✔ started ssh service | 2026-06-17 07:00:44,169


...

ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


INFO - ✔ connected to remote kernel | 2026-06-17 07:00:53,514


WARNING - ! remote kernel environment has a different set of libraries installed | 2026-06-17 07:00:53,515


Success: connected to remote kernel via r450.modal.host:44339


In [ ]:
%remote !nvidia-smi

Wed Jun 17 07:00:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       On  |   00000000:F4:00.0 Off |                    0 |
| N/A   27C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|


|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+


I'll similarly create another function so the sandbox can be turned off in one line as well.

In [ ]:
#| export
def gpu_off(
      sb:modal.sandbox.Sandbox  # Sandbox object from gpu_on
):
    "Restore local kernel, disconnect remote, and teardown sandbox. Must be called from a %%local cell."
    unset_sticky()
    stop_remote()
    log.info('✔ unlinked from remote kernel')
    sb.terminate()
    log.info('✔ terminated sandbox')

In [ ]:
gpu_off(sb)

INFO - ✔ unlinked from remote kernel | 2026-06-17 07:01:26,152


INFO - ✔ terminated sandbox | 2026-06-17 07:01:26,214


Code cells will now run locally.
Code cells will now run locally.
Shutting down remote kernel


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()